In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install duckdb -q
import duckdb
con = duckdb.connect()

In [ ]:
con.execute("DROP TABLE IF EXISTS raw_utilization")
con.execute("DROP TABLE IF EXISTS stg_utilization")
con.execute("DROP TABLE IF EXISTS clean_utilization")
con.execute("DROP TABLE IF EXISTS fact_drug_uptake")

con.execute("""
    CREATE TABLE raw_utilization AS
    SELECT * FROM 'global_pharmacy_sales_2020_2025_daily_dataset.csv'
""")

con.execute("""
    CREATE TABLE stg_utilization AS
    SELECT DISTINCT
        CAST(date AS DATE) AS date,
        region, country, category, medicine, age_group,
        CAST(units_sold AS DOUBLE) AS units_sold,
        CAST(unit_price AS DOUBLE) AS unit_price,
        CAST(stock_level AS DOUBLE) AS stock_level,
        CAST(covid_flag AS INTEGER) AS covid_flag
    FROM raw_utilization
""")

con.execute("""
    CREATE TABLE clean_utilization AS
    SELECT * FROM stg_utilization
    WHERE units_sold IS NOT NULL AND units_sold >= 0 AND stock_level IS NOT NULL
""")

con.execute("""
    CREATE TABLE fact_drug_uptake AS
    WITH first_seen AS (
        SELECT medicine, region, MIN(date) AS launch_proxy_date
        FROM clean_utilization
        GROUP BY medicine, region
    ),
    monthly_sales AS (
        SELECT medicine AS drug_id, region, DATE_TRUNC('month', date) AS month, SUM(units_sold) AS utilization
        FROM clean_utilization
        GROUP BY medicine, region, DATE_TRUNC('month', date)
    )
    SELECT m.drug_id, m.region, m.month, m.utilization,
           DATEDIFF('month', DATE_TRUNC('month', f.launch_proxy_date), m.month) AS months_since_launch
    FROM monthly_sales m
    LEFT JOIN first_seen f ON m.drug_id = f.medicine AND m.region = f.region
    ORDER BY m.drug_id, m.region, m.month
""")

print("fact_drug_uptake rows:", con.execute("SELECT COUNT(*) FROM fact_drug_uptake").fetchone()[0])

import pandas as pd

region_assumptions = pd.DataFrame({
    "region": ["Africa","Europe","East Asia","North America","South Asia","Middle East","South America","Oceania"],
    "population_millions": [1460, 745, 1680, 375, 1970, 470, 435, 45],
    "chronic_prevalence_pct": [28.5, 40.0, 29.0, 32.0, 27.0, 38.0, 30.0, 29.0],
    "cough_cold_episodes_per_year": [1.4, 2.0, 1.8, 2.3, 1.5, 1.9, 1.8, 2.1],
    "antibiotic_episodes_per_year": [0.4, 0.5, 0.5, 0.6, 0.4, 0.5, 0.5, 0.5],
    "antipyretic_episodes_per_year": [0.8, 1.0, 1.0, 1.1, 0.9, 1.0, 1.0, 1.0],
    "vitamin_usage_pct": [15, 40, 30, 55, 12, 25, 25, 35],
    "sourcing_note": [
        "chronic: sourced (Africa meta-analysis)",
        "chronic: estimated",
        "chronic: sourced (WHO Western Pacific)",
        "cough_cold: sourced (GBD high-income NA)",
        "estimated",
        "chronic: sourced (WHO Eastern Mediterranean)",
        "estimated",
        "chronic: sourced (WHO Western Pacific)"
    ]
})

con.register("region_assumptions_df", region_assumptions)
con.execute("CREATE OR REPLACE TABLE region_assumptions AS SELECT * FROM region_assumptions_df")
con.execute("SELECT * FROM region_assumptions").df()

In [ ]:
con.execute("DROP TABLE IF EXISTS modeling_dataset")

con.execute("""
    CREATE TABLE modeling_dataset AS
    WITH drug_category AS (
        SELECT DISTINCT medicine, category FROM clean_utilization
    ),
    joined AS (
        SELECT
            f.drug_id, dc.category, f.region, f.month, f.utilization, f.months_since_launch,
            r.population_millions, r.chronic_prevalence_pct, r.cough_cold_episodes_per_year,
            r.antibiotic_episodes_per_year, r.antipyretic_episodes_per_year, r.vitamin_usage_pct
        FROM fact_drug_uptake f
        LEFT JOIN drug_category dc ON f.drug_id = dc.medicine
        LEFT JOIN region_assumptions r ON f.region = r.region
    )
    SELECT
        *,
        CASE category
            WHEN 'Chronic' THEN population_millions * 1000000 * chronic_prevalence_pct / 100
            WHEN 'Cough_Cold' THEN population_millions * 1000000 * cough_cold_episodes_per_year
            WHEN 'Antibiotic' THEN population_millions * 1000000 * antibiotic_episodes_per_year
            WHEN 'Antipyretic' THEN population_millions * 1000000 * antipyretic_episodes_per_year
            WHEN 'Vitamin' THEN population_millions * 1000000 * vitamin_usage_pct / 100
        END AS eligible_population_proxy,
        CASE category
            WHEN 'Chronic' THEN 'prevalence-based (point prevalence of condition)'
            WHEN 'Vitamin' THEN 'usage-rate-based (propensity to use)'
            ELSE 'incidence-based (expected annual episodes, not a population count)'
        END AS denominator_type
    FROM joined
""")

con.execute("ALTER TABLE modeling_dataset ADD COLUMN uptake_rate_proxy DOUBLE")
con.execute("UPDATE modeling_dataset SET uptake_rate_proxy = utilization / eligible_population_proxy")

print("rows:", con.execute("SELECT COUNT(*) FROM modeling_dataset").fetchone()[0])  # should be 5760
con.execute("""
    SELECT drug_id, category, region, month, utilization, eligible_population_proxy, denominator_type, uptake_rate_proxy
    FROM modeling_dataset ORDER BY drug_id, region, month LIMIT 15
""").df()

In [ ]:
import numpy as np
np.random.seed(42)  # reproducibility — same synthetic assignment every run

drugs = con.execute("SELECT DISTINCT drug_id FROM modeling_dataset").df()["drug_id"].tolist()

# Reimbursement tier: assigned per drug (a drug's reimbursement status is a national decision— simplification

reimbursement_tiers = pd.DataFrame({
    "drug_id": drugs,
    "reimbursement_tier": np.random.choice(["High", "Medium", "Low"], size=len(drugs), p=[0.4, 0.4, 0.2])
})

# Competitor entry: assigned per drug — a random month between month 6 and month 30 of the
# observed window, reflecting typical generic/competitor entry delay after a product is established
competitor_entry = pd.DataFrame({
    "drug_id": drugs,
    "competitor_entry_month_offset": np.random.randint(6, 30, size=len(drugs))
})

print(reimbursement_tiers)
print(competitor_entry)

In [ ]:
con.register("reimbursement_tiers_df", reimbursement_tiers)
con.register("competitor_entry_df", competitor_entry)

con.execute("DROP TABLE IF EXISTS modeling_dataset_final")

con.execute("""
    CREATE TABLE modeling_dataset_final AS
    SELECT
        m.*,
        rt.reimbursement_tier,
        ce.competitor_entry_month_offset,
        CASE WHEN m.months_since_launch >= ce.competitor_entry_month_offset THEN 1 ELSE 0 END AS competitor_present
    FROM modeling_dataset m
    LEFT JOIN reimbursement_tiers_df rt ON m.drug_id = rt.drug_id
    LEFT JOIN competitor_entry_df ce ON m.drug_id = ce.drug_id
""")

print("rows:", con.execute("SELECT COUNT(*) FROM modeling_dataset_final").fetchone()[0])  # should be 5760
con.execute("""
    SELECT drug_id, region, month, months_since_launch, reimbursement_tier, competitor_entry_month_offset, competitor_present
    FROM modeling_dataset_final ORDER BY drug_id, region, month LIMIT 30
""").df()

In [ ]:
con.execute("DROP TABLE IF EXISTS modeling_dataset")
con.execute("ALTER TABLE modeling_dataset_final RENAME TO modeling_dataset")

In [ ]:
print("Final check:", con.execute("SELECT COUNT(*) FROM modeling_dataset").fetchone()[0])